# Estimacion DSGE Smets & Wouters - Argentina (MCMC)

Este notebook ejecuta una prueba MCMC con **5000 replicas por cadena** (2 cadenas = 10,000 draws total)
para obtener la **Tabla 1A** con distribuciones posteriores completas.

- Carga el posterior mode pre-computado (`argmodel_mode.mat`)
- Corre Metropolis-Hastings desde ese mode
- Genera Tabla 1A con: Post. Mode, Post. Mean, HPD inf (5%), HPD sup (95%)

**Tiempo estimado**: ~30-60 minutos

## 1. Setup y Configuracion

In [1]:
# Imports
import numpy as np
import pandas as pd
import os
import sys
from pathlib import Path

# Agregar directorio padre al path
sys.path.append(str(Path.cwd().parent))

from src import DynareInterface

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Imports completados")

Imports completados


In [2]:
# Configurar rutas
os.environ['OCTAVE_EXECUTABLE'] = r'C:\Program Files\GNU Octave\Octave-10.3.0\mingw64\bin\octave-cli.exe'

DYNARE_PATH = r'C:\dynare\6.5\matlab'
MODEL_PATH = Path.cwd().parent / 'model'
DATA_PATH = Path.cwd().parent / 'data'
OUTPUT_PATH = Path.cwd().parent / 'output'

print(f"Model path: {MODEL_PATH}")
print(f"Mode file exists: {(MODEL_PATH / 'argmodel_mode.mat').exists()}")

Model path: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECTMT11\DG\model
Mode file exists: True


## 2. Verificar datos y mode file

In [3]:
# Verificar que existen los archivos necesarios
mat_file = MODEL_PATH / 'argmodel_data.mat'
mode_file = MODEL_PATH / 'argmodel_mode.mat'

if not mat_file.exists():
    print("ERROR: argmodel_data.mat no existe. Ejecutar primero tables_argentina_stst.ipynb")
else:
    print(f"Datos OK: {mat_file}")

if not mode_file.exists():
    print("ERROR: argmodel_mode.mat no existe. Ejecutar primero argmodel.mod con mode_compute=4")
else:
    print(f"Mode file OK: {mode_file}")

Datos OK: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECTMT11\DG\model\argmodel_data.mat
Mode file OK: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECTMT11\DG\model\argmodel_mode.mat


## 3. Definicion de parametros Tabla 1A

In [4]:
# TABLA 1A - PARAMETROS ESTRUCTURALES
TABLE_1A_PARAMS = {
    # Parameter: (Dynare name, Prior Distr, Prior Mean, Prior SD, Description)
    'phi':       ('csadjcost', 'Normal',  4.00, 1.50, 'Costo ajuste inversion'),
    'sigma_c':   ('csigma',    'Normal',  1.50, 0.37, 'Aversion al riesgo'),
    'h':         ('chabb',     'Beta',    0.70, 0.10, 'Formacion de habito'),
    'xi_w':      ('cprobw',    'Beta',    0.50, 0.10, 'Prob. Calvo salarios'),
    'sigma_l':   ('csigl',     'Normal',  2.00, 0.75, 'Elasticidad oferta laboral'),
    'xi_p':      ('cprobp',    'Beta',    0.50, 0.10, 'Prob. Calvo precios'),
    'iota_w':    ('cindw',     'Beta',    0.50, 0.15, 'Indexacion salarios'),
    'iota_p':    ('cindp',     'Beta',    0.50, 0.15, 'Indexacion precios'),
    'psi':       ('czcap',     'Beta',    0.50, 0.15, 'Utilizacion capacidad'),
    'Phi':       ('cfc',       'Normal',  1.25, 0.12, 'Costo fijo'),
    'r_pi':      ('crpi',      'Normal',  1.50, 0.25, 'Taylor: inflacion'),
    'rho':       ('crr',       'Beta',    0.75, 0.10, 'Taylor: suavizamiento'),
    'r_y':       ('cry',       'Normal',  0.12, 0.05, 'Taylor: output gap'),
    'r_Delta_y': ('crdy',      'Normal',  0.12, 0.05, 'Taylor: crecimiento output'),
    'pi_bar':    ('constepinf','Gamma',   0.62, 0.10, 'Inflacion estado estacionario'),
    'beta_const':('constebeta','Gamma',   0.25, 0.10, '100(beta^-1 - 1)'),
    'l_bar':     ('constelab', 'Normal',  0.00, 2.00, 'Horas estado estacionario'),
    'gamma_bar': ('ctrend',    'Normal',  0.40, 0.10, 'Tendencia crecimiento'),
    'alpha':     ('calfa',     'Normal',  0.30, 0.05, 'Participacion capital'),
}

print(f"Tabla 1A: {len(TABLE_1A_PARAMS)} parametros estructurales definidos")

Tabla 1A: 19 parametros estructurales definidos


## 4. Limpiar corridas previas y ejecutar MCMC

In [5]:
# Limpiar carpeta de corridas MCMC previas (evita prompts interactivos de Dynare)
import shutil
import time

mcmc_dir = MODEL_PATH / 'argmodel_mcmc'
if mcmc_dir.exists():
    print(f"Limpiando corrida previa: {mcmc_dir}")
    try:
        shutil.rmtree(mcmc_dir)
        print("Carpeta eliminada OK")
    except PermissionError:
        # Si Octave tiene archivos bloqueados, renombrar
        backup = MODEL_PATH / f'argmodel_mcmc_old_{int(time.time())}'
        mcmc_dir.rename(backup)
        print(f"No se pudo eliminar (archivos bloqueados), renombrada a: {backup.name}")
else:
    print("No hay corridas previas para limpiar")

No hay corridas previas para limpiar


In [6]:
# Inicializar interfaz Dynare
di = DynareInterface(DYNARE_PATH, str(MODEL_PATH))
print("Interfaz Dynare inicializada")

    _pyeval at line 57 column 10

    _pyeval at line 57 column 10

    _pyeval at line 57 column 10

Interfaz Dynare inicializada


In [7]:
# Ejecutar modelo MCMC
print("Ejecutando argmodel_mcmc.mod...")
print("(5000 replicas x 2 cadenas - esto puede tardar 30-60 minutos)\n")

di.run_model('argmodel_mcmc.mod')

print("\nMCMC completado!")

Ejecutando argmodel_mcmc.mod...
(5000 replicas x 2 cadenas - esto puede tardar 30-60 minutos)


Step 1: Closing Octave session to release file locks...
Waiting for Windows to release file handles...

Step 2: Cleaning up directories...
Searching for directories to clean up...
No directories found to clean up.

Step 3: Starting fresh Octave session...
    _pyeval at line 57 column 10

    _pyeval at line 57 column 10

    _pyeval at line 57 column 10

Octave session ready

Step 4: Running Dynare estimation...
Command: dynare argmodel_mcmc nograph
(This may take several minutes...)

Starting Dynare (version 6.5).
Calling Dynare with arguments: nograph
Starting preprocessing of the model file ...
Found 40 equation(s).
Evaluating expressions...
Computing static model derivatives (order 1).
Normalizing the static model...
Finding the optimal block decomposition of the static model...
11 block(s) found:
  9 recursive block(s) and 2 simultaneous block(s).
  the largest simultaneous block has 1

Oct2PyError: Octave evaluation error:
error: operator *: nonconformant arguments (op1 is 36x36, op2 is 0x0)
error: called from:
    posterior_sampler_initialization at line 205, column 25
    posterior_sampler at line 59, column 3
    dynare_estimation_1 at line 444, column 17
    dynare_estimation at line 105, column 5
    driver at line 1288, column 1
    dynare at line 306, column 5

## 5. Verificar convergencia MCMC

In [ ]:
# Verificar tasa de aceptacion
try:
    acc_rate = di.oc.eval('oo_.MH_accept_rate', nout=1)
    if hasattr(acc_rate, 'flatten'):
        acc_rate = acc_rate.flatten()
    print("Tasa de aceptacion por cadena:")
    for i, rate in enumerate(acc_rate):
        status = "OK" if 0.20 <= rate <= 0.35 else "AJUSTAR mh_jscale"
        print(f"  Cadena {i+1}: {rate:.4f} ({status})")
    print(f"\nPromedio: {np.mean(acc_rate):.4f}")
    print("(Optimo: 0.23-0.30)")
except Exception as e:
    print(f"No se pudo obtener tasa de aceptacion: {e}")

## 6. Extraer resultados

In [ ]:
# Extraer posterior mode
params_df = di.get_parameters()
estimated_params = dict(zip(params_df['parameter'], params_df['value']))
print(f"Extraidos {len(estimated_params)} parametros (posterior mode)")

In [ ]:
# Extraer resultados MCMC (media posterior e intervalos HPD)
def extract_mcmc_results(di):
    """Extrae media posterior e intervalos HPD 90% de los resultados MCMC."""
    results = {
        'available': False,
        'param_means': {},
        'param_hpd_inf': {},
        'param_hpd_sup': {},
    }
    
    try:
        has_mcmc = di.oc.eval(
            'isfield(oo_, "posterior_mean") && isfield(oo_.posterior_mean, "parameters")',
            nout=1
        )
        
        if not has_mcmc:
            print("ERROR: Resultados MCMC no disponibles")
            return results
        
        results['available'] = True
        print("Resultados MCMC disponibles!")
        
        # Obtener nombres de parametros estimados
        n_params = int(di.oc.eval('length(fieldnames(oo_.posterior_mean.parameters))', nout=1))
        param_names = []
        for i in range(n_params):
            name = di.oc.eval(f'deblank(M_.param_names{{estim_params_.param_vals({i+1},1)}})', nout=1)
            param_names.append(str(name).strip())
        
        # Medias posteriores
        means = di.oc.eval('struct2array(oo_.posterior_mean.parameters)', nout=1).flatten()
        for i, name in enumerate(param_names):
            if i < len(means):
                results['param_means'][name] = float(means[i])
        
        # Intervalos HPD
        has_hpd = di.oc.eval(
            'isfield(oo_, "posterior_hpdinf") && isfield(oo_.posterior_hpdinf, "parameters")',
            nout=1
        )
        if has_hpd:
            hpd_inf = di.oc.eval('struct2array(oo_.posterior_hpdinf.parameters)', nout=1).flatten()
            hpd_sup = di.oc.eval('struct2array(oo_.posterior_hpdsup.parameters)', nout=1).flatten()
            for i, name in enumerate(param_names):
                if i < len(hpd_inf):
                    results['param_hpd_inf'][name] = float(hpd_inf[i])
                    results['param_hpd_sup'][name] = float(hpd_sup[i])
            print(f"Intervalos HPD 90% extraidos para {len(results['param_hpd_inf'])} parametros")
        
        print(f"Media posterior extraida para {len(results['param_means'])} parametros")
        
    except Exception as e:
        print(f"Error extrayendo MCMC: {e}")
    
    return results

mcmc = extract_mcmc_results(di)

## 7. Generar Tabla 1A

In [ ]:
def create_table_1A(param_defs, estimated_params, mcmc):
    """Crea Tabla 1A con posterior mode, media e intervalos HPD."""
    rows = []
    
    for param_symbol, values in param_defs.items():
        dynare_name, prior_distr, prior_mean, prior_sd, description = values
        
        post_mode = estimated_params.get(dynare_name, np.nan)
        post_mean = np.nan
        hpd_inf = np.nan
        hpd_sup = np.nan
        
        if mcmc['available']:
            post_mean = mcmc['param_means'].get(dynare_name, np.nan)
            hpd_inf = mcmc['param_hpd_inf'].get(dynare_name, np.nan)
            hpd_sup = mcmc['param_hpd_sup'].get(dynare_name, np.nan)
        
        rows.append({
            'Parametro': param_symbol,
            'Nombre Dynare': dynare_name,
            'Descripcion': description,
            'Prior Distr.': prior_distr,
            'Prior Mean': prior_mean,
            'Prior SD': prior_sd,
            'Post. Mode': post_mode,
            'Post. Mean': post_mean,
            'HPD inf (5%)': hpd_inf,
            'HPD sup (95%)': hpd_sup,
        })
    
    return pd.DataFrame(rows)

table_1A = create_table_1A(TABLE_1A_PARAMS, estimated_params, mcmc)
print("Tabla 1A generada")

In [ ]:
# Mostrar Tabla 1A completa
print("=" * 110)
print("TABLA 1A - DISTRIBUCION PRIOR Y POSTERIOR DE PARAMETROS ESTRUCTURALES")
print("Modelo DSGE Smets & Wouters - Argentina (2004Q2-2025Q3)")
print("MCMC: 5000 replicas x 2 cadenas")
print("=" * 110)

display_cols = ['Parametro', 'Prior Distr.', 'Prior Mean', 'Prior SD', 
                'Post. Mode', 'Post. Mean', 'HPD inf (5%)', 'HPD sup (95%)']

def fmt(x):
    if pd.isna(x):
        return 'N/A'
    return f'{x:.4f}'

print(table_1A[display_cols].to_string(index=False, float_format=fmt))

## 8. Exportar y guardar

In [ ]:
# # Exportar tabla a CSV
# tables_dir = OUTPUT_PATH / 'tables'
# tables_dir.mkdir(parents=True, exist_ok=True)

# output_file = tables_dir / 'table_1A_argentina_mcmc.csv'
# table_1A.to_csv(output_file, index=False)
# print(f"Tabla 1A guardada en: {output_file}")

In [ ]:
# Comparar Post. Mode vs Post. Mean
if mcmc['available']:
    print("\nComparacion Mode vs Mean:")
    print(f"{'Parametro':<12} {'Mode':>10} {'Mean':>10} {'Diff %':>10}")
    print("-" * 45)
    for _, row in table_1A.iterrows():
        mode = row['Post. Mode']
        mean = row['Post. Mean']
        if pd.notna(mode) and pd.notna(mean) and mode != 0:
            diff_pct = (mean - mode) / abs(mode) * 100
            print(f"{row['Parametro']:<12} {mode:>10.4f} {mean:>10.4f} {diff_pct:>+9.1f}%")
else:
    print("Sin resultados MCMC para comparar")

## 9. Cleanup

In [ ]:
# Cerrar sesion Octave
di.close()
print("Sesion Octave cerrada")
print("\nPrueba MCMC completada!")